## Encounter simulation: 1 player vs 1 enemy

### Using for environments:
* RL card games (https://github.com/datamllab/rlcard)
* Chess petting zoo (https://pettingzoo.farama.org/environments/classic/chess/)

---

## Strategy overview
1. Define the player and the enemy creature and pull out all base stats and full action list from data
2. Determine initative order (does enemy or player go first)

Start of encounter: ticker= 3 actions each 
3. Setup environment and define the current state

START OF EACH ROUND
* Distance of player and enemy to eachother -> determines if melee or range
* Current player stats -> HP, AC, other relevant stats, and calculate and current buffs or conditions that would impact those stats; current spell slots
* Current enemy stats -> same and player

4. Based on the state, filter what actions are possible
* Filter action list based on 1) distance to the enemy (melee/range), 2) Conditions that need to be met (i.e. off-gaurd for sneak attack), 3) action economy (spell slots, cost of each action)

5. Choose actions up to the 3 action cost (either RANDOMLY, MAX DAMAGE, or using Deep q-learning) and calculate 1) Success of hit, 2) total damage dealt, 3) impact on status of enemy (give a condition)
* For one round get both the player and the enemy action choices and results

6. Calculate the reward for that round: Goal is to optimize the action combinations that results in the LEAST number of rounds for an encounter to end
** Alternatively -> run the action choices until end of encounter (so either the enemy or player runs down to 0)
* Reward strategy 1= (0/1/-1) action success + total damage dealt + bonus if end of round (+ if player wins, -1 if monster wins)
* Reward strategy 2= Just the result of the whole encounter (+1 player wins, -1 monster wins) + total number of rounds for the encounter

7. Update the state based on actions (health, status) and repeat simulation util one of the two creatures health = 0 

As a result, the data I will have: 
1. How DIFFICULT is the monster for each player, based on the number of rounds it took to defeat and whether the moster defeated the player
2. What is the combination of attacks (for both enemy and player) that is MOST EFFECTIVE against the opponent

I would repeat the simulation for each player-monster combination for 20 rounds of combat (number arbitrary), then have a major dataset that has:
1. Difficulty ranking for each player-monster based on (data distribution) of number rounds and relative difficulty --> and the spread for how likely the minumum/optimal round encounter is for each combination
2. A list of effective attacks for the monster against specific players -> helpful for a DM running a campaign to know what attack actions to pick
3.(stretch goal/ fall back) Features of the attacks and monster details that could be used in a linear regression to predict if a certain factor of a monster makes it more or less difficult (i.e. a range fire monster is more challenging for a fighter)

The stretch goal:
* Use the difficulty of multple players in combination to see what the difficulty of a group of monsters would be
* Use a battle map layout

----

In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import gymnasium
#Import libraries 
import pandas as pd
import numpy as np
import requests
import matplotlib as plt
import seaborn as sns
import matplotlib.pyplot as plt


import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import re
import random

import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=SettingWithCopyWarning)

### Import data for player

In [53]:
### pull in player base stats

In [54]:
def open_clean_actionlist(filepath):
    actionlist = pd.read_csv(filepath)
    #clean up nan calues 
    actionlist= actionlist.replace('x', 0).fillna(0)

    #add a column that calculates changes in location 
    loc_change_list =[]
    for action in range(len(actionlist)):
        action_choice = actionlist.iloc[action]
        if action_choice['action'] == 'move towards target':
            loc_change_list.append('+speed')
        elif action_choice['action'] == 'move away from target':
            loc_change_list.append('-speed')
        elif action_choice['action'] == 'flank target':
            loc_change_list.append('+speed')
        else:
            loc_change_list.append(0)
    
    actionlist['location_change'] = loc_change_list

    #Change data types if needed 
    actionlist['phy_attack_require']= actionlist['phy_attack_require'].astype(int)

    return actionlist

char_actlist = open_clean_actionlist('../build data/rogue_actionlist.csv')
char_actlist.sample(3)

,Cost,target_cond_requirement,phy_attack_require,action,action type,trait,Description,HP bonus,save effect,AC effect,...,range,number die,damage die,additional damage,damage type,duration (round),cool down (round),crit success,crit failure,location_change
0,1,0,0,move towards target,movement,speed,25ft (5 squares),0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,+speed
15,0,melee attack,0,nimble dodge,buff,0,"trigger- creature attack, dodge = +2 AC",0,0,2,...,0,0,0,0,0,0,0,0.0,0.0,0
6,1,0,3,rapier attack (third),melee,dexterity,0,0,0,0,...,0,1,6,3,P,0,0,0.0,0.0,0


In [55]:
## import character build basic stats
build_skill = pd.read_csv('../build data/all_build_base_skills.csv')
char_skill = build_skill[build_skill['Unnamed: 0']== 'halfling rogue dexterity']
#rename columns to match monster format 
char_skill= char_skill.rename(columns={'hit points':'HP', 'armor class':'AC', 'Unnamed: 0':'character build'})
#make into a dictionary for eaiser analysis
char_skill_dict = char_skill.iloc[0].to_dict()

#view dictionary 
char_skill_dict

{'character build': 'halfling rogue dexterity',
 'ancestry': 'halfling',
 'class': 'rogue dexterity',
 'level': 2,
 'HP': 24,
 'speed': 25,
 'perception': 8,
 'fortitude': 5,
 'reflex': 9,
 'will': 8,
 'AC': 18,
 'unarmored': 2,
 'light': 2,
 'medium': 0,
 'heavy': 0,
 'unarmed': 2,
 'simple': 2,
 'martial': 0,
 'advanced': 0,
 'other': 0,
 'magical tradition': 0,
 'spell attack': 0,
 'spell dc': 10}

### Import data for enemy

In [56]:
monster_build = pd.read_csv('../build data/monster_data_short.csv')
## Clean up the main dataset 
monster_build['name']= monster_build['name'].str.lower()

### Pull out only the stat relevant columns
stat_col = ['name', 'Creature Level', 'Perception',
       'HP', 'AC', 'Fort', 'Ref', 'Will', 'Fort DC', 'Ref DC', 'Will DC',
       'speed_Land','Attack Op']

monster_build= monster_build[stat_col]

#rename columns if needed 
monster_build = monster_build.rename(columns={'speed_Land':'speed'})

#fil in nan with zeros 
monster_build = monster_build.fillna(0)

#make all column names lowercase 
monster_build.columns = monster_build.columns.str.lower()

#view dataframe
monster_build.head(3)

,name,creature level,perception,hp,ac,fort,ref,will,fort dc,ref dc,will dc,speed,attack op
0,abandoned zealot,6,14,75,22,10,14,16,-2.0,0.0,0.0,0,False
1,adlet,10,18,180,30,20,22,16,0.0,0.0,0.0,40,False
2,arbiter,1,7,22,16,5,7,7,-1.0,0.0,0.0,20,False


In [57]:
monster_choice= 'kobold scout'

In [58]:
#Check all monster options to make sure monster of interest is in the list 
monster_list = monster_build['name'].astype('str').unique().tolist()
#make it all lower case 
monster_list = [x.lower() for x in monster_list]
monster_check = monster_choice
monster_name = [x for x in monster_list if re.search(monster_check, x)]
monster_name

#pull out the row for that monster stat 
choice_monster_build = monster_build[monster_build['name'].str.lower() == monster_name[0]]
#make all build options to a dictionary for easier analysis
mon_stat_dict= choice_monster_build.iloc[0].to_dict()

mon_stat_dict

{'name': 'kobold scout',
 'creature level': 1,
 'perception': 8,
 'hp': 16,
 'ac': 18,
 'fort': 5,
 'ref': 9,
 'will': 6,
 'fort dc': -3.0,
 'ref dc': 0.0,
 'will dc': 0.0,
 'speed': '25',
 'attack op': False}

In [59]:
### Pull out all action list items for the chosen monster
mon_actionlist = pd.read_csv('../build data/monster_all_actionlist.csv')

## Filter full monster action list for the monster of interest
enemy_actionlist = mon_actionlist[mon_actionlist['monster_name'] == monster_choice]

enemy_actionlist

,Unnamed: 0,monster_name,monster_level,action_term,action_name,damage_type,attack_range,action_cost,attack_type,damage_dice_calculations,attack_roll_mod,spell_DC,spell_slot_level,casting time,target,area,duration,saving throw,spell attack,conditions
1585,1769,kobold scout,1,Attack 1,shortsword,['piercing'],5,1,melee,"[['1', '6', 0]]",9.0,0,0,0,0,0,0,0,0,0
1586,1770,kobold scout,1,Attack 2,crossbow,['piercing'],120,2,range,"[['1', '8', 0]]",9.0,0,0,0,0,0,0,0,0,0


----

### Define starting state for the encounter
* player and enemy starting stats
* Intiative order for encounter 

In [60]:
## Stats that need to be included in the state: distance location, health (HP), defense (AC), speed, spell slot counter, condition effects

In [61]:
player_cond_effects='none'
player_state = {'location': char_skill_dict['speed'], 
'hp': char_skill_dict['HP'],
'ac': char_skill_dict['AC'],
'speed': char_skill_dict['speed'],
'percep': char_skill_dict['perception'],
'fort': char_skill_dict['fortitude'],
'ref': char_skill_dict['reflex'],
'will': char_skill_dict['will'], 
'spell_dc': char_skill_dict['spell dc'],
'condition': player_cond_effects}

In [62]:
mon_cond_effects='none'
enemy_state= {'location': mon_stat_dict['speed'], 
'hp': mon_stat_dict['hp'],
'ac': mon_stat_dict['ac'],
'speed': mon_stat_dict['speed'],
'percep': mon_stat_dict['perception'],
'fort': mon_stat_dict['fort'],
'ref': mon_stat_dict['ref'],
'will': mon_stat_dict['will'], 
'spell_dc': 0,
'condition': mon_cond_effects}

### Roll for Initiative!

In [95]:
#character initative= dice roll + perception 
char_intve = random.randint(1, 20) + player_state['percep'] 
#print("character initative: ", char_intve)
mon_intve = random.randint(1, 20) + enemy_state['percep'] 
#print("monster initative: ", mon_intve)

----

### Encounter strategy one: Random action selection
* Add feature that updates player/enemy state stats based on conditions. And then conditions are reset between each round
* Filter out action options once they are used in previous round -> so you wont get 3 non-attack actions the same back to back
* Create monster round code based on monster action list format

In [92]:
def update_action_by_location(distance_calculation, action_list):
    agent_distance= distance_calculation
    actlist_filter = action_list
    #If the player and enemy are within 5 feet of eachother -> can use melee actions and movement actions
    if agent_distance<=5:
       #actlist_filter = actlist_filter[actlist_filter['action type'] != 'range'] 
        ### start with only physical attacks, add in buff and spells later
        actlist_filter = actlist_filter[(actlist_filter['action type'] =='melee')|((actlist_filter['action type'] =='movement'))]
    #Else the player can only use range and movement 
    else:
        #actlist_filter = actlist_filter[actlist_filter['action type'] != 'melee']
        ### start with only physical attacks, add in buff and spells later
        actlist_filter = actlist_filter[(actlist_filter['action type'] =='range')|((actlist_filter['action type'] =='movement'))]
    return actlist_filter

######################################################################################
### Simulate one round encounter using random selection 
def one_round_player(action_list, player_dict, enemy_dict, round_number):
    #define necessary variables
    current_turn_player= 'player'
    player_state= player_dict 
    enemy_state = enemy_dict
    char_actlist= action_list
    #Create a list to save the results 
    enc_result=[]
    
    #define starting state values 
    ### Check on location -> filter based on location options
    agent_distance = abs(int(player_state['location']) - int(enemy_state['location']))
    target_ac = enemy_state['ac']
    target_health = enemy_state['hp']
    
    #Start the counter for the number of actions available
    action_num= 3
    MAP_counter= 1
    
    ### Simulate one round of combat ----------------------------------------------
    while action_num >0:
    #### Selection action from avaiable actions ###
        #Filter based on player distance 
        actlist_filter= update_action_by_location(agent_distance, char_actlist)
        #Filter based on action economy 
        actlist_filter= actlist_filter[(actlist_filter['phy_attack_require']==MAP_counter) | (actlist_filter['phy_attack_require']==0)]
        #filter based on condition or other requirements ###################
        
        ### Random Choice ###
        #Pick an avaiable action by random 
        attack_choice = actlist_filter.sample(n=1).iloc[0]
    
        ### Simulate the results of that action
        #calculate the chance of hitting 
        roll_calc = int(attack_choice['attack roll']) + random.randint(1, 20)
        #roll_calc = 30
        crit_level = max(int(attack_choice['attack roll']) + 20, int(target_ac) + 10)
        crit_fail_level = int(attack_choice['attack roll']) + 1
        
        #results for a normal hit
        if roll_calc >= target_ac and roll_calc <= crit_level:
            #print("sucess") #calculate total damage 
            die_max = int(attack_choice['damage die'])
            if die_max ==0: 
                die_max = 4
            num_dmg_die= int(float(attack_choice['number die']))
            total_damage = (num_dmg_die * random.randint(1, die_max)) + int(attack_choice['additional damage'])
            attack_sucess = 1
        #results for a critical hit 
        elif roll_calc >=target_ac and roll_calc >= crit_level:
           # print("crit sucess") #calculate total damage 
            die_max = int(attack_choice['damage die'])
            if die_max ==0: 
                die_max = 4
            num_dmg_die= int(float(attack_choice['number die']))
            total_damage = (num_dmg_die * random.randint(1, die_max)) + (num_dmg_die * random.randint(1, die_max))+ int(attack_choice['additional damage'])
            attack_sucess = 2
        #results for a critical fail 
        elif roll_calc < target_ac and roll_calc <= crit_fail_level:
            #print("crit fail") #calculate total damage 
            total_damage = 0
            attack_sucess = -1
        #results for a normal fail
        elif roll_calc < target_ac:
            #print('fail')
            total_damage = 0
            attack_sucess = 0
            #print('total damage =', total_damage)
        #store read errors as failures
        else:
            #print('other')
            total_damage = 0
            attack_sucess = 0
            #print('total damage =', total_damage)
        
        ## Now calculate the state changes from the roll and save results 
        
        ######calculate the 'reward' or change in state based on these results #####
        #Current health or AC changes based on the results 
        target_health = target_health - total_damage
        
        #calculate action reward
        attack_reward= attack_sucess*total_damage
        if target_health <=0:
            attack_reward= attack_reward + 100
    
        ### Update encounter states:
        ## Update location 
        if attack_choice['location_change'] != 0:
            total_location_change = int(attack_choice['location_change'].replace('speed', str(player_state['speed'])))
            agent_distance= agent_distance- total_location_change
        ## Add in conditions
        enemy_state['condition']= (char_actlist[char_actlist['action'] == 'flank target']['target effect'].tolist())
       
         #Save the results in the results_list 
        enc_result.append([round_number, current_turn_player, action_num, attack_choice['action'], agent_distance, attack_sucess, total_damage, target_health, attack_reward])
        
        #Calculate the action cost 
        action_num = action_num - attack_choice['Cost']
    
        #calculate the MAP cost if it was a melee or range attack
        if attack_choice['action type'] == 'melee' or attack_choice['action type'] == 'range':
            MAP_counter= +1
    # #View result of choices 
    # results_col= ['player', 'combat_round', 'action_num', 'attack_choice', 'attack_sucess', 'total_damage', 'target_health', 'attack_reward']
     
    ### After the round is complete
    ## Update player/enemy states based on action results
    enemy_state['hp'] = target_health
    ### Change state based on conditions --> Will need to code a conditions dictionary here
    
    ##After one round of player combat ==> Return the results and each of the player and enemy state
    
    return enc_result, player_state, enemy_state

In [94]:
####
#Starting encounter reset
####
#Reset player starting state 
player_cond_effects='none'
player_state = {'location': char_skill_dict['speed'], 
'hp': char_skill_dict['HP'],
'ac': char_skill_dict['AC'],
'speed': char_skill_dict['speed'],
'percep': char_skill_dict['perception'],
'fort': char_skill_dict['fortitude'],
'ref': char_skill_dict['reflex'],
'will': char_skill_dict['will'], 
'spell_dc': char_skill_dict['spell dc'],
'condition': player_cond_effects}

#reset enemy starting state
mon_cond_effects='none'
enemy_state= {'location': mon_stat_dict['speed'], 
'hp': mon_stat_dict['hp'],
'ac': mon_stat_dict['ac'],
'speed': mon_stat_dict['speed'],
'percep': mon_stat_dict['perception'],
'fort': mon_stat_dict['fort'],
'ref': mon_stat_dict['ref'],
'will': mon_stat_dict['will'], 
'spell_dc': 0,
'condition': mon_cond_effects}

## Define player starting position ### Define starting locations as one movement away from eachother 
player_state.update({'location': -player_state['speed']})
enemy_state.update({'location': enemy_state['speed']})

### Roll for intiative ### 
#character initative= dice roll + perception 
char_intve = random.randint(1, 20) + player_state['percep'] 
#print("character initative: ", char_intve)
mon_intve = random.randint(1, 20) + enemy_state['percep'] 
#print("monster initative: ", mon_intve)

####
# Start encounter: random selection 
####

#Save results to a main dataframe 
full_enc_results= pd.DataFrame()

### Simulate the results from one full encounter
# if char_intve >= mon_intve:
round_number = 1
while enemy_state['hp'] >0 or player_state['hp'] >0:
    round_actions, player_state, enemy_state = one_round_player(char_actlist, player_state, enemy_state, round_number)
    full_enc_results= pd.concat([full_enc_results, pd.DataFrame(round_actions)])
    print(f'{round_number}: monster HP {enemy_state['hp']}')

    #End the loop once the monster dies
    if enemy_state['hp'] < 0: 
        break

    # mon_round, char_health = round_physical_attack_result('kobold', monster_actlist, round_number, char_health, char_ac)
    # full_enc_results= pd.concat([full_enc_results, mon_round])
    # print(f'{round_number}: character HP {char_health}')

    # if char_health < 0: 
    #     break

    round_number +=1

# elif char_intve < mon_intve:
#     round_number = 1
#     while mon_health >0 or char_health >0:
#         mon_round, char_health = round_physical_attack_result('kobold', monster_actlist, round_number, char_health, char_ac)
#         full_enc_results= pd.concat([full_enc_results, mon_round])
#         print(f'{round_number}: character HP {char_health}')

#         if char_health < 0: 
#             break

#         char_round, mon_health = round_physical_attack_result('rogue', char_actlist, round_number, mon_health, mon_ac)
#         full_enc_results= pd.concat([full_enc_results, char_round])
#         print(f'{round_number}: monster HP {mon_health}')

#         if mon_health < 0: 
#             break
#         round_number +=1

full_enc_results

1: monster HP 16
2: monster HP 10
3: monster HP 5
4: monster HP 5
5: monster HP 5
6: monster HP 5
7: monster HP 5
8: monster HP 0
9: monster HP -6


,0,1,2,3,4,5,6,7,8
0,1,player,3,Throw Dagger attack (first),50,0,0,16,0
1,1,player,2,flank target,25,0,0,16,0
2,1,player,1,move away from target,50,0,0,16,0
0,2,player,3,Throw Dagger attack (first),50,1,6,10,6
1,2,player,2,move away from target,75,0,0,10,0
2,2,player,1,move towards target,50,0,0,10,0
0,3,player,3,Throw Dagger attack (first),50,1,5,5,5
1,3,player,2,move towards target,25,0,0,5,0
2,3,player,1,flank target,0,1,0,5,0
0,4,player,3,move towards target,25,0,0,5,0


----